# Deep Learning Project 1: ResNet on CIFAR-10 with at most 5M parameters

This notebook trains a modified ResNet from scratch on the competition's CIFAR-10 training data
and writes `submission.csv` for the Kaggle competition
[cifar-10-res-net-architecture-challenge-dl](https://www.kaggle.com/competitions/cifar-10-res-net-architecture-challenge-dl).

It is designed for **Google Colab with a GPU** (Runtime > Change runtime type > GPU).
The model, data pipeline, and training loop live in `src/`; this notebook runs them and shows the results.

Steps:
1. Setup: clone the repo, install packages, mount Google Drive for checkpoints
2. Download the competition data with the Kaggle API
3. Inspect the data and the candidate architectures (parameter counts)
4. Screening runs to compare architectures
5. Final training run, learning curves, and final accuracy and parameter count
6. Predict the evaluation images and write `submission.csv`

## 1. Setup

In [ ]:
import os, sys

REPO_URL = "https://github.com/Vedant0703/Deep-Learning-2026.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not os.path.exists("src"):
    if not os.path.exists("/content/repo"):  # already cloned if the kernel was only restarted
        !git clone -q $REPO_URL /content/repo
    %cd /content/repo
elif os.path.basename(os.getcwd()) == "notebooks":  # running locally from notebooks/
    %cd ..

!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no NVIDIA GPU"

In [ ]:
%pip install -q torchsummary kaggle six  # kaggle 2.2.5 imports six without declaring it

In [ ]:
# Checkpoints go to Google Drive so a Colab disconnect doesn't lose a run (resume with --resume).
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = "/content/drive/MyDrive/dl-project1/runs"
else:
    RUN_ROOT = "runs"
os.makedirs(RUN_ROOT, exist_ok=True)
print("Runs saved under", RUN_ROOT)

## 2. Download the competition data

You need a Kaggle API token from [kaggle.com/settings/api](https://www.kaggle.com/settings/api); the cell asks for it
(the prompt also works when this notebook runs on a Colab kernel inside VS Code or Cursor).
You must also have accepted the competition rules on the competition page, or the download is refused.

In [ ]:
COMPETITION = "cifar-10-res-net-architecture-challenge-dl"

if not os.path.exists("data/train_labels.csv"):
    if "KAGGLE_API_TOKEN" not in os.environ and not os.path.exists(os.path.expanduser("~/.kaggle/access_token")):
        from getpass import getpass
        os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token: ")
    !kaggle competitions download $COMPETITION -p data
    !unzip -q -o data/$COMPETITION.zip -d data
!ls data

## 3. Data and model overview

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torchsummary import summary

from src.data import CLASSES, load_split
from src.models import CONFIGS, PARAM_BUDGET, build_model, count_params

train_images, train_labels, _ = load_split("data", "train")
test_images, _, test_ids = load_split("data", "test")
print("train:", train_images.shape, "| evaluation:", test_images.shape)
print("train class counts:", np.bincount(train_labels).tolist())
for name, a in [("train", train_images), ("evaluation", test_images)]:
    a = a / 255.0
    print(f"{name:>10} mean {a.mean((0, 1, 2)).round(3)} std {a.std((0, 1, 2)).round(3)}")

In [ ]:
# One training example per class (top row) and unlabeled evaluation images (bottom row).
fig, axes = plt.subplots(2, 10, figsize=(15, 3.4))
for c in range(10):
    axes[0, c].imshow(train_images[np.flatnonzero(train_labels == c)[0]])
    axes[0, c].set_title(CLASSES[c], fontsize=9)
    axes[1, c].imshow(test_images[c])
for ax in axes.flat:
    ax.axis("off")
axes[1, 0].set_title("evaluation", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
rows = []
for name, cfg in CONFIGS.items():
    rows.append({"model": name, "channels": cfg["channels"], "blocks": cfg["blocks"],
                 "params": count_params(build_model(name))})
df = pd.DataFrame(rows)
df["under 5M budget"] = df.params <= PARAM_BUDGET
df.style.format({"params": "{:,}"})

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
summary(build_model("resnet_a").to(device), (3, 32, 32), device=device)

## 4. Screening: compare architectures

Each candidate gets the same 40-epoch recipe on a stratified 45k/5k train/val split:
SGD with Nesterov momentum (peak LR 0.1, 3 warmup epochs, then cosine decay), batch 256, weight decay 5e-4,
label smoothing 0.1, augmentation of random crop + flip + TrivialAugment + random erasing, and mixed precision.

In [ ]:
SCREEN_EPOCHS = 40
for m in CONFIGS:
    out = f"{RUN_ROOT}/screen_{m}"
    !python -m src.train --model $m --epochs $SCREEN_EPOCHS --out "$out" --resume

In [ ]:
def load_history(run):
    with open(f"{RUN_ROOT}/{run}/history.json") as f:
        return pd.DataFrame(json.load(f))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for m in CONFIGS:
    h = load_history(f"screen_{m}")
    axes[0].plot(h.epoch, h.train_loss, label=m)
    axes[1].plot(h.epoch, h.val_acc, label=f"{m} (best {h.val_acc.max():.2f}%)")
axes[0].set(title="Training loss", xlabel="epoch"); axes[1].set(title="Validation accuracy (%)", xlabel="epoch")
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Final training run

Train the best screening architecture for longer. Keeping the 5k validation split gives a held-out accuracy
for the report. Setting `FINAL_VAL_SIZE = 0` trains on all 50k images for the final submission,
but then no held-out accuracy is reported.

In [ ]:
FINAL_MODEL = "resnet_a"   # set to the best screening model
FINAL_EPOCHS = 200
FINAL_VAL_SIZE = 5000
FINAL_RUN = f"final_{FINAL_MODEL}"
out = f"{RUN_ROOT}/{FINAL_RUN}"
!python -m src.train --model $FINAL_MODEL --epochs $FINAL_EPOCHS --val-size $FINAL_VAL_SIZE --out "$out" --resume

In [ ]:
h = load_history(FINAL_RUN)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(h.epoch, h.train_loss, label="train")
axes[1].plot(h.epoch, h.train_acc, label="train")
if h.val_acc.notna().any():
    axes[0].plot(h.epoch, h.val_loss, label="val")
    axes[1].plot(h.epoch, h.val_acc, label="val")
axes[2].plot(h.epoch, h.lr)
axes[0].set(title="Loss", xlabel="epoch"); axes[1].set(title="Accuracy (%)", xlabel="epoch")
axes[2].set(title="Learning rate", xlabel="epoch")
for ax in axes: ax.grid(alpha=0.3)
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
from src.data import build_loaders
from src.predict import load_model
from src.train import evaluate, get_device

device = get_device()
final_model = load_model(f"{RUN_ROOT}/{FINAL_RUN}/final.pt", device)
n_params = count_params(final_model)
print(f"Trainable parameters: {n_params:,} (budget {PARAM_BUDGET:,})")
assert n_params <= PARAM_BUDGET

if FINAL_VAL_SIZE > 0:
    _, val_loader = build_loaders("data", val_size=FINAL_VAL_SIZE)
    use_amp = device.type == "cuda"
    _, acc = evaluate(final_model, val_loader, device, use_amp)
    _, acc_tta = evaluate(final_model, val_loader, device, use_amp, tta=True)
    print(f"Final validation accuracy: {acc:.2f}% (with flip TTA: {acc_tta:.2f}%)")

## 6. Predict the evaluation images and write `submission.csv`

In [ ]:
ckpt = f"{RUN_ROOT}/{FINAL_RUN}/final.pt"
!python -m src.predict --ckpt "$ckpt" --out submission.csv
pd.read_csv("submission.csv").head()

In [ ]:
# Submitting counts toward the 5-per-day team limit. Uncomment to submit from Colab.
# !kaggle competitions submit $COMPETITION -f submission.csv -m "resnet_a, 200 epochs"